In [ ]:
# ============================================================
# PHASE 4 — MODÉLISATION PRÉDICTIVE & INTERPRÉTABILITÉ
# Objectif : Prédire si une commande aura un avis négatif (score ≤ 2)
# ============================================================

# ── 0. IMPORTS ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, roc_auc_score, roc_curve,
                              confusion_matrix, ConfusionMatrixDisplay,
                              precision_recall_curve, average_precision_score)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

import shap
shap.initjs()

sns.set_theme(style='whitegrid')

In [ ]:
# ── 1. CHARGEMENT DES DONNÉES (depuis Phase 3) ──────────────

df  = pd.read_csv('../data/processed/master_table.csv', low_memory=False)
rfm = pd.read_csv('../data/processed/rfm_clusters.csv')

print(f"Master table : {df.shape}")
print(f"RFM clusters : {rfm.shape}")

In [ ]:
# ── 2. FEATURE ENGINEERING COMPLET ──────────────────────────

# TARGET : avis négatif = 1 (score ≤ 2), sinon 0
df['target'] = (df['review_score'] <= 2).astype(int)
print(f"Taux d'avis négatifs : {df['target'].mean()*100:.1f}%")

# ---- Features temporelles ----
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')
df['order_hour']     = df['order_purchase_timestamp'].dt.hour
df['order_dow']      = df['order_purchase_timestamp'].dt.dayofweek
df['order_month_nb'] = df['order_purchase_timestamp'].dt.month

# ---- Features livraison ----
df['delivery_days'] = pd.to_numeric(df.get('delivery_days', np.nan), errors='coerce')
df['delay_days']    = pd.to_numeric(df.get('delay_days',    np.nan), errors='coerce')
df['is_late']       = (df['delay_days'] > 0).astype(int)

# ---- Features commande ----
df['revenue'] = df['price'].fillna(0) + df['freight_value'].fillna(0)
df['freight_ratio'] = df['freight_value'] / (df['price'] + 1e-6)

# ---- Encodage catégoriel ----
le = LabelEncoder()
df['customer_state_enc'] = le.fit_transform(df['customer_state'].fillna('unknown'))

top_cats = df['product_category_name_english'].value_counts().head(20).index
df['category_top20'] = df['product_category_name_english'].apply(lambda x: x if x in top_cats else 'other')
df['category_enc'] = le.fit_transform(df['category_top20'].fillna('unknown'))

df['payment_type_enc'] = le.fit_transform(df['payment_type'].fillna('unknown'))

# ---- Merge RFM features ----
df = df.merge(rfm[['customer_unique_id','recency','frequency','monetary','cluster']],
              on='customer_unique_id', how='left')

# ---- Sélection features finales ----
FEATURES = [
    'price', 'freight_value', 'freight_ratio', 'revenue',
    'delivery_days', 'delay_days', 'is_late',
    'order_hour', 'order_dow', 'order_month_nb',
    'customer_state_enc', 'category_enc', 'payment_type_enc',
    'payment_installments',
    'recency', 'frequency', 'monetary', 'cluster'
]

df_model = df[FEATURES + ['target']].dropna()
print(f"\nDataset de modélisation : {df_model.shape}")
print(f"Taux de positifs : {df_model['target'].mean()*100:.1f}%")

In [ ]:
# ── 3. SPLIT TRAIN / TEST ────────────────────────────────────

X = df_model[FEATURES]
y = df_model['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"Taux positifs — Train : {y_train.mean():.3f}  |  Test : {y_test.mean():.3f}")

In [ ]:
# ── 5. VALIDATION CROISÉE (K-FOLD = 5) ──────────────────────

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:25s} AUC-ROC CV : {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# ── 6. ENTRAÎNEMENT FINAL & ÉVALUATION ──────────────────────

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_proba)
    ap  = average_precision_score(y_test, y_proba)
    
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
        'auc': auc, 'ap': ap
    }
    
    print(f"\n{'='*50}")
    print(f"MODÈLE : {name}")
    print(f"  AUC-ROC = {auc:.4f}  |  Avg Precision = {ap:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Positif','Négatif']))

In [ ]:
# ── 7. COURBES ROC COMPARATIVES ──────────────────────────────

plt.figure(figsize=(9, 7))
colors = ['steelblue', 'tomato', 'mediumseagreen']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC = {res['auc']:.3f})")

plt.plot([0,1],[0,1],'k--', lw=1, label='Aléatoire')
plt.xlabel('Taux de Faux Positifs'); plt.ylabel('Taux de Vrais Positifs')
plt.title('Courbes ROC — Comparaison des 3 Modèles', fontsize=14, fontweight='bold')
plt.legend(loc='lower right'); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/phase4_roc_curves.png', dpi=150)
plt.show()

In [ ]:
# ── 8. PRECISION-RECALL CURVES ───────────────────────────────

plt.figure(figsize=(9, 7))
for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    plt.plot(rec, prec, lw=2, color=color, label=f"{name} (AP = {res['ap']:.3f})")

plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Courbes Precision-Recall', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/phase4_precision_recall.png', dpi=150)
plt.show()

In [ ]:
# ── 9. MATRICE DE CONFUSION — MEILLEUR MODÈLE ────────────────

best_name = max(results, key=lambda n: results[n]['auc'])
best      = results[best_name]
print(f"Meilleur modèle : {best_name}  (AUC = {best['auc']:.4f})")

cm = confusion_matrix(y_test, best['y_pred'])
disp = ConfusionMatrixDisplay(cm, display_labels=['Positif','Négatif'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title(f'Matrice de Confusion — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/phase4_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── 10. FEATURE IMPORTANCE (Random Forest) ──────────────────

rf_model = results['Random Forest']['model']
importances = pd.DataFrame({
    'feature': FEATURES,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(9, 7))
plt.barh(importances['feature'], importances['importance'], color='steelblue')
plt.title('Feature Importance — Random Forest (Top 15)', fontsize=13, fontweight='bold')
plt.xlabel('Importance (Gini)')
plt.tight_layout()
plt.savefig('../reports/figures/phase4_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# ── 11. SHAP VALUES — INTERPRÉTABILITÉ ──────────────────────

print("Calcul des SHAP values (peut prendre 1-2 min)...")

# On prend XGBoost ou RF selon le meilleur modèle
if best_name == 'XGBoost':
    explainer = shap.TreeExplainer(best['model'])
else:
    explainer = shap.TreeExplainer(results['Random Forest']['model'])

# Échantillon pour la vitesse
X_sample = X_test.sample(min(1500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)

# Pour les classifieurs binaires RF, shap_values est une liste [class0, class1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

In [ ]:
# 11.1 SHAP Summary Plot (Global)
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_sample, feature_names=FEATURES, show=False)
plt.title("SHAP Summary Plot — Impact Global des Features", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/phase4_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 11.2 SHAP Bar Plot (importance moyenne absolue)
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_sample, feature_names=FEATURES,
                  plot_type='bar', show=False)
plt.title("SHAP — Importance Moyenne Absolue", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/phase4_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 11.3 SHAP Force Plot — Explication locale (1 prédiction)
idx = 0  # première observation du sample
print(f"\nExplication locale — observation #{idx}")
print(f"Vraie valeur : {y_test.iloc[idx]} | Probabilité prédite : {best['y_proba'][y_test.index.get_loc(X_sample.index[idx])]:.3f}")

force_plot = shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    sv[idx],
    X_sample.iloc[idx],
    feature_names=FEATURES,
    matplotlib=True, show=False
)
plt.savefig('../reports/figures/phase4_shap_force_local.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 11.4 SHAP Dependence Plot — delay_days (feature clé)
plt.figure(figsize=(9, 6))
shap.dependence_plot('delay_days', sv, X_sample,
                     feature_names=FEATURES, interaction_index='is_late', show=False)
plt.title("SHAP Dependence — delay_days (couleur = is_late)", fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/phase4_shap_dependence.png', dpi=150)
plt.show()

In [ ]:
# ── 12. TABLEAU COMPARATIF FINAL ─────────────────────────────

summary = pd.DataFrame({
    'Modèle': list(results.keys()),
    'AUC-ROC': [results[n]['auc'] for n in results],
    'Avg Precision': [results[n]['ap'] for n in results],
    'CV AUC (mean)': [cv_results[n].mean() for n in results],
    'CV AUC (std)':  [cv_results[n].std()  for n in results],
}).set_index('Modèle').round(4)

print("\n", "="*55)
print("TABLEAU COMPARATIF — PERFORMANCE DES MODÈLES")
print("="*55)
print(summary.to_string())

summary.to_csv('../reports/phase4_model_comparison.csv')
print("\n✅ Sauvegardé → reports/phase4_model_comparison.csv")

In [ ]:
# ── 13. SAUVEGARDE DU MEILLEUR MODÈLE ───────────────────────

import joblib, os
os.makedirs('../models', exist_ok=True)

best_model_obj = results[best_name]['model']
joblib.dump(best_model_obj, f'../models/best_model_{best_name.replace(" ","_").lower()}.pkl')
print(f"✅ Meilleur modèle sauvegardé : models/best_model_{best_name.replace(' ','_').lower()}.pkl")

In [ ]:
# ── 14. RÉSUMÉ EXÉCUTIF PHASE 4 ─────────────────────────────

print("""
╔══════════════════════════════════════════════════════════╗
║           RÉSUMÉ — PHASE 4 MODÉLISATION                  ║
╠══════════════════════════════════════════════════════════╣
║  Problème   : Classification binaire (avis négatif ≤ 2)  ║
║  3 modèles  : Logistic Regression, Random Forest, XGB    ║
║  Meilleur   : {:<38s}║
║  AUC-ROC    : {:<38s}║
║  Top features (SHAP) :                                   ║
║    1. delay_days    — retard de livraison                ║
║    2. delivery_days — durée totale de livraison          ║
║    3. freight_ratio — rapport frais port/prix            ║
║    4. monetary      — valeur client RFM                  ║
║  → Prêt pour Phase 5 (Dashboard) & Phase 6 (A/B Test)   ║
╚══════════════════════════════════════════════════════════╝
""".format(best_name, f"{best['auc']:.4f}"))